# Gemini Embeddings

## Embedding models

The [Gemini API](https://ai.google.dev/gemini-api/docs/embeddings) offers two models that generate text embeddings:

- `gemini-embedding-2` — recommended; multimodal (text, images, video, audio, PDFs)
- `gemini-embedding-001` — stable; text-only (legacy)


In [ ]:
from dotenv import load_dotenv

load_dotenv()

import os
import getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("GEMINI_API_KEY")

In [ ]:
# Uncomment if running on google colab
# !pip install -U -q google-genai

In [ ]:
from google import genai

gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

result = client.models.embed_content(
        model="gemini-embedding-2",
        contents="What is the meaning of life?")

print(result.embeddings)

Leveraging [task types](https://ai.google.dev/gemini-api/docs/embeddings#task-types) to generate optimized embeddings for specific tasks.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=gemini_api_key)

result = client.models.embed_content(
        model="gemini-embedding-2",
        contents="What is the meaning of life?",
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)
print(result.embeddings)

# Setting up a Simple Embeddings DB

In [ ]:
from IPython.display import Markdown

with open("./assets-resources/prompt-engineering-summary-report.md") as f:
    sample_1 = f.read()

Markdown(sample_1)

In [ ]:
with open("./assets-resources/gemini-report-summary.md") as f:
    sample_2 = f.read()

Markdown(sample_2)

In [ ]:
# inspired by the docs: https://ai.google.dev/gemini-api/docs/structured-output?lang=python
from google import genai
from pydantic import BaseModel


class Section(BaseModel):
    title: str
    content: str

def extract_sections(report_text: str) -> list[Section]:
    """
    Extract main sections and their content from a report using Gemini.
    
    Args:
        report_text: The text content of the report to analyze
        
    Returns:
        List of Section objects containing titles and content
    """
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents=f'Extract the main sections and their content from the following report:\n\n{report_text}',
        config={
            'response_mime_type': 'application/json',
            'response_schema': list[Section],
        },
    )
    
    # Return the parsed sections
    return response.parsed

# Example usage:
sections1 = extract_sections(sample_1)
sections2 = extract_sections(sample_2)
print(sections1[0].title)
print(sections2[0].title)

In [ ]:
documents = [
    {"title": section.title, "content": section.content}
    for section in sections1 + sections2
]

In [ ]:
documents

In [ ]:
import pandas as pd

df = pd.DataFrame(documents)

df.columns = ['Title', 'Text']
df

In [ ]:
# Get the embeddings of each text and add to an embeddings column in the dataframe
def embed_fn(title, text):
    return client.models.embed_content(
        model="gemini-embedding-2",
        contents=f"Title: {title}. Content:\n\n{text}",
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")).embeddings[0].values

df['Embeddings'] = df.apply(lambda row: embed_fn(row['Title'], row['Text']), axis=1)
df

Now the procedure to setup document search & Q&A is:

![](./assets-resources/embeddings1.png)

and then for the search and Q&A:

![](./assets-resources/embeddings2.png)

In [ ]:
query = "what is the most critical prompting technique?"

embedding_model = "gemini-embedding-2"

request = client.models.embed_content(
        model=embedding_model,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
)

request

In [ ]:
import numpy as np

def find_best_passage(query, dataframe):
  """
  Compute the distances between the query and each document in the dataframe
  using the dot product.
  """
  query_embedding = client.models.embed_content(
                    model=embedding_model,
                    contents=query,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
  ).embeddings[0].values
  dot_products = np.dot(np.stack(dataframe['Embeddings']), query_embedding)
  idx = np.argmax(dot_products)
  return dataframe.iloc[idx]['Text'] # Return text from index with max value
  
passage = find_best_passage(query, df)
passage

In [ ]:
query = "what are the best practices for prompting gemini models?"

embedding_model = "gemini-embedding-2"

passage = find_best_passage(query, df)
passage

In [ ]:
# this function was taken from this notebook: https://github.com/google/generative-ai-docs/blob/main/site/en/gemini-api/tutorials/document_search.ipynb
import textwrap

def make_prompt(query, relevant_passage):
    escaped = relevant_passage.replace("'", "").replace('"', "").replace("\n", " ")
    prompt = textwrap.dedent("""You are a helpful and informative bot that answers questions using text from the reference passage included below. \
                            Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. \
                            However, you are talking to a non-technical audience, so be sure to break down complicated concepts and \
                            strike a friendly and converstional tone. \
                            If the passage is irrelevant to the answer, you may ignore it.
                            QUESTION: '{query}'
                            PASSAGE: '{relevant_passage}'

    ANSWER:
  """).format(query=query, relevant_passage=escaped)
    
    return prompt

prompt = make_prompt(query, passage)
print(prompt)

In [ ]:
answer = client.models.generate_content(
    model='gemini-3.8-flash',
    contents=prompt
)

answer.text

Check out more examples in the [gemini docs](https://ai.google.dev/gemini-api/docs/embeddings).